In [ ]:
!pip install --upgrade scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 56.1 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1


In [ ]:
import xgboost as xgb
import shap
import pandas as pd
import numpy as np
from typing import Union, Dict, Optional, Tuple, Set, List
from math import factorial
import time
from copy import copy
from tqdm import tqdm
from collections import defaultdict
from sklearn.metrics import accuracy_score, f1_score
import sklearn
import math

import warnings
warnings.filterwarnings("ignore", category=FutureWarning, module=r"sklearn\..*")

In [ ]:
# Useful if you run this on google colab and downloaded the data into your drive.
# If you run the notebook in other environment remove these lines and change the 'pd.read_csv()' function in this notebook to read from
# where you saved you data
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# import woodelf from Python file in the drive
!cp /content/drive/MyDrive/...../woodelf.py /content/

import woodelf

# PDIVs Code

In [ ]:
from itertools import combinations

def all_subsets(s):
    subsets = []
    for k in range(len(s) + 1):
        for subset in combinations(s, k):
            subsets.append(set(subset))
    return subsets

class PDIV(woodelf.CubeMetric):
    INTERACTION_VALUE = True
    def calc_metric(
        self, s_plus: Set, s_minus: Set
    ) -> Dict[str, float]:
        if len(s_plus & s_minus) > 0:
            return {}

        pdivs = {}
        for sm in all_subsets(s_minus):
            s = tuple(s_plus | sm)
            pdivs[s] = (-1) ** (len(sm))
        return pdivs

# Fraud Data + Model

In [ ]:
transactions_train = pd.read_parquet('drive/MyDrive/PATH_TO_DATA_CREATED_IN_THE_Preprocess_Data_NOTEBOOK.parquet') # columns are train_features + ['isFraud']
transactions_test = pd.read_parquet('drive/MyDrive/PATH_TO_DATA_CREATED_IN_THE_Preprocess_Data_NOTEBOOK.parquet') # columns are train_features + ['isFraud']

train_features = [f for f in transactions_train.columns if f != 'isFraud']
fraud_train = transactions_train[train_features]
fraud_test = transactions_test[train_features]

In [ ]:
def train_hist_gradient_boosting_model(X, y, max_depth=6):
    gradient_boosting_model = sklearn.ensemble.HistGradientBoostingRegressor(
        max_iter=100,
        max_depth=max_depth,
        max_leaf_nodes=None,
        random_state=42,
        min_samples_leaf=1
    )
    gradient_boosting_model.fit(X, y)
    return gradient_boosting_model

models = {depth: train_hist_gradient_boosting_model(fraud_train, transactions_train['isFraud'], depth) for depth in tqdm(range(1, 11))}

for depth, model in models.items():
    y_pred = model.predict(transactions_test[train_features])
    print(f"Depth {depth}. Accuracy: {accuracy_score(transactions_test['isFraud'], y_pred.round())}, F1 score: {f1_score(transactions_test['isFraud'], y_pred.round())}")

100%|██████████| 10/10 [05:21<00:00, 32.11s/it]


Depth 1. Accuracy: 0.9702645036746029, F1 score: 0.27347952006619775
Depth 2. Accuracy: 0.9706116435804518, F1 score: 0.32875652678398765
Depth 3. Accuracy: 0.9718901344532123, F1 score: 0.37452901281085155
Depth 4. Accuracy: 0.9720594709926508, F1 score: 0.3938280675973549
Depth 5. Accuracy: 0.9729400209977309, F1 score: 0.41721371261852663
Depth 6. Accuracy: 0.9715006604125038, F1 score: 0.4133844545137679
Depth 7. Accuracy: 0.9622802858400785, F1 score: 0.35218845426784934
Depth 8. Accuracy: 0.9724150777254716, F1 score: 0.4347675225537821
Depth 9. Accuracy: 0.9640583195041826, F1 score: 0.3814658312691243
Depth 10. Accuracy: 0.9624919565143767, F1 score: 0.3726989521382045


## Woodelf PDP computation



In [ ]:
# RAM crash as depth 9
depth_times = []
for depth in models:
    start_time = time.time()
    woodelf.calculate_background_metric(
        models[depth], consumer_data=fraud_train.head(10_000), background_data=fraud_train, metric=PDIV(), global_importance=True, GPU=False
    )
    depth_times.append(str(round(time.time() - start_time, 1)))
    print(f"\n Any Order PDIVs n=10,000, Depth {depth} took: {time.time() - start_time} sec")
    print(" & ".join(depth_times))

Preprocessing the trees: 100%|██████████| 100/100 [00:00<00:00, 249.90it/s]


cache misses: 1, cache used: 199


Computing the values: 100%|██████████| 100/100 [00:00<00:00, 2777.32it/s]



 Any Order PDIVs n=10,000, Depth 1 took: 0.45078063011169434 sec
0.5


Preprocessing the trees: 100%|██████████| 100/100 [00:00<00:00, 130.60it/s]


...

Preprocessing the trees: 100%|██████████| 100/100 [05:26<00:00,  3.27s/it]


cache misses: 663, cache used: 28679


Computing the values: 100%|██████████| 100/100 [15:14<00:00,  9.15s/it]



 Any Order PDIVs n=10,000, Depth 9 took: 1242.2768952846527 sec
0.5 & 0.9 & 1.9 & 4.0 & 9.4 & 24.6 & 77.5 & 274.7 & 1242.3


Preprocessing the trees:  19%|█▉        | 19/100 [05:41<23:04, 17.10s/it]

In [ ]:
del transactions_test, transactions_train, fraud_test, fraud_train

# KDD-Cup 1999: Intrusion Detection Dataset

In [ ]:
detection_data = pd.read_parquet("drive/MyDrive/PATH_TO_DATA_CREATED_IN_THE_Preprocess_Data_NOTEBOOK.parquet")
unlabeled_data = pd.read_parquet("drive/MyDrive/PATH_TO_DATA_CREATED_IN_THE_Preprocess_Data_NOTEBOOK.parquet")
small_test_data = pd.read_parquet("drive/MyDrive/PATH_TO_DATA_CREATED_IN_THE_Preprocess_Data_NOTEBOOK.parquet")

detection_train_features_names = [f for f in detection_data.columns if f != "target"]
detection_trainset = detection_data[detection_train_features_names]

In [ ]:
def train_hist_gradient_boosting_model(X, y, max_depth=6):
    gradient_boosting_model = sklearn.ensemble.HistGradientBoostingRegressor(
        max_iter=100,
        max_depth=max_depth,
        max_leaf_nodes=None,
        random_state=42,
        min_samples_leaf=1
    )
    gradient_boosting_model.fit(X, y)
    return gradient_boosting_model

detection_models = {depth: train_hist_gradient_boosting_model(detection_trainset, detection_data['target'], depth) for depth in tqdm(range(1, 11))}

100%|██████████| 10/10 [12:09<00:00, 72.91s/it]


In [ ]:
for depth, model in detection_models.items():
    y_pred = model.predict(small_test_data[detection_train_features_names])
    print(f"Depth {depth}. Accuracy: {accuracy_score(small_test_data['target'], (y_pred > 0.5).astype(int))}, F1 score: {f1_score(small_test_data['target'], (y_pred > 0.5).astype(int))}")

Depth 1. Accuracy: 0.9140208790820149, F1 score: 0.9439458029571932
Depth 2. Accuracy: 0.9235183857453807, F1 score: 0.9501801122560107
Depth 3. Accuracy: 0.9241324763928765, F1 score: 0.9506096093267611
Depth 4. Accuracy: 0.9254185301049098, F1 score: 0.9514943552619748
Depth 5. Accuracy: 0.9267753167711049, F1 score: 0.9524181602802889
Depth 6. Accuracy: 0.9264120065974556, F1 score: 0.9521739857240769
Depth 7. Accuracy: 0.92774628732369, F1 score: 0.953082038059647
Depth 8. Accuracy: 0.925659665175916, F1 score: 0.9516638166394207
Depth 9. Accuracy: 0.9261676563921692, F1 score: 0.9520201914679348
Depth 10. Accuracy: 0.9257689797414389, F1 score: 0.9517647402925704


In [ ]:
del detection_data, unlabeled_data, small_test_data

## Woodelf PDP computation


In [ ]:
# RAM crashed at depth 9
depth_times = []
for depth in detection_models:
    start_time = time.time()
    woodelf.calculate_background_metric(
        detection_models[depth], consumer_data=detection_trainset.head(10_000), background_data=detection_trainset, metric=PDIV(), global_importance=True, GPU=False
    )
    depth_times.append(str(round(time.time() - start_time, 1)))
    print(f"\n Any Order PDIVs n=10,000, Depth {depth} took: {time.time() - start_time} sec")
    print(" & ".join(depth_times))

Preprocessing the trees: 100%|██████████| 100/100 [00:02<00:00, 47.36it/s]


cache misses: 1, cache used: 199


Computing the values: 100%|██████████| 100/100 [00:00<00:00, 4202.71it/s]



 Any Order PDIVs n=10,000, Depth 1 took: 2.1481685638427734 sec
2.1


Preprocessing the trees: 100%|██████████| 100/100 [00:04<00:00, 24.24it/s]


...

Preprocessing the trees: 100%|██████████| 100/100 [05:50<00:00,  3.51s/it]


cache misses: 815, cache used: 15344


Computing the values: 100%|██████████| 100/100 [03:02<00:00,  1.82s/it]



 Any Order PDIVs n=10,000, Depth 9 took: 532.9257724285126 sec
2.1 & 4.2 & 8.3 & 18.7 & 36.0 & 61.8 & 105.0 & 202.0 & 532.9


Preprocessing the trees:  71%|███████   | 71/100 [11:50<05:19, 11.01s/it]